In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 16 — O THRESHOLD DE DECISÃO: AJUSTANDO A SENSIBILIDADE DO MODELO

> "Todo modelo é um juiz. E todo juiz tem um padrão de dúvida razoável. A arte não está em eliminar a dúvida, mas em ajustar o padrão ao crime em questão."
> — Um Estatístico Forense

Nove falsos negativos. O número era uma pedra no meu sapato. Meu modelo, apesar da acurácia alta, ainda deixava passar o equivalente a nove pacientes com câncer. A questão era: **como reduzir isso?**

Refleti sobre como o modelo decide. Nos bastidores, o SVM calcula um **escore de decisão**; por padrão, se o escore aponta mais para "benigno" do que para "maligno", ele decide benigno. Esse ponto de corte — o *threshold* — trata falsos positivos e falsos negativos com o mesmo peso. Mas eles **não** têm o mesmo peso. E se eu dissesse ao modelo: "seja mais cauteloso — na menor suspeita, classifique como maligno"? Alguns dos nove falsos negativos virariam falsos positivos, mais gerenciáveis. Era uma troca — mas uma troca que eu faria conscientemente.

## O Threshold e o Custo dos Erros

Ajustar o *threshold* move o modelo ao longo do *trade-off* Precisão-Recall. Baixar o corte (ser mais agressivo em prever "Maligno") **aumenta o Recall** e reduz Falsos Negativos, ao custo de mais Falsos Positivos (menor Precisão). Subir o corte faz o oposto.

Mas qual *threshold* escolher? Em vez de um número arbitrário, vamos usar os **custos clínicos** que definimos lá no Capítulo 1: `CUSTO_FN = 100` e `CUSTO_FP = 10` — deixar um câncer passar é dez vezes pior que um alarme falso. Para cada *threshold* possível, calculamos o **custo total** = 100·FN + 10·FP e escolhemos o que o **minimiza**. Assim, a decisão deixa de ser estética e passa a ser guiada pela realidade clínica.

#### Código 16.1: Escolhendo o Threshold Pelo Custo Clínico


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict
from oncoclassify_utils import X_train, y_train, CUSTO_FN, CUSTO_FP

def info_mutua(X, y):
    return mutual_info_classif(X, y, random_state=42)

modelo = Pipeline([("selecao", SelectKBest(info_mutua, k=25)),
                   ("escala", StandardScaler()),
                   ("svm", SVC(kernel="rbf", C=10, gamma=0.01, random_state=42))])

# escore de decisao (>0 -> Benigno). Prevemos Maligno se o escore <= t.
score = cross_val_predict(modelo, X_train, y_train, cv=5, method="decision_function")
eh_maligno = (y_train == 0).astype(int).values

def avalia_threshold(t):
    prev_mal = (score <= t).astype(int)
    fn = int(((prev_mal == 0) & (eh_maligno == 1)).sum())
    fp = int(((prev_mal == 1) & (eh_maligno == 0)).sum())
    tp = int(((prev_mal == 1) & (eh_maligno == 1)).sum())
    recall = tp / (tp + fn)
    custo  = CUSTO_FN * fn + CUSTO_FP * fp
    return recall, fn, fp, custo

thresholds = np.linspace(-2.5, 2.5, 251)
custos = [avalia_threshold(t)[3] for t in thresholds]
t_otimo = thresholds[int(np.argmin(custos))]

rec_pad, fn_pad, fp_pad, custo_pad = avalia_threshold(0.0)      # padrao
rec_ot,  fn_ot,  fp_ot,  custo_ot  = avalia_threshold(t_otimo)  # custo minimo

print(f"Threshold PADRAO (t=0.00): recall={rec_pad:.3f}  FN={fn_pad}  FP={fp_pad}  custo={custo_pad}")
print(f"Threshold OTIMO  (t={t_otimo:.2f}): recall={rec_ot:.3f}  FN={fn_ot}  FP={fp_ot}  custo={custo_ot}")

Threshold PADRAO (t=0.00): recall=0.949  FN=9  FP=2  custo=920
Threshold OTIMO  (t=0.80): recall=0.989  FN=2  FP=24  custo=440


![Recall, precisão e custo em função do threshold](media/figura_16_1.png)

O resultado é concreto e honesto. No *threshold* padrão, o modelo tinha **9 Falsos Negativos** e 2 Falsos Positivos, custo total **920**. Movendo o corte para o ponto de **custo mínimo** (t = 0,80), os Falsos Negativos caem de **9 para 2** — capturamos **sete** dos cânceres que antes escapavam. O preço: os Falsos Positivos sobem de 2 para 24, e a precisão cai. Mas o **custo clínico total despenca de 920 para 440**: para o problema que temos, alarmar 24 pacientes saudáveis é preferível a perder 7 diagnósticos de câncer.

O recall maligno sobe de 94,9% para **98,9%**. Não é mágica nem um número inventado — é a consequência mensurável de decidir, conscientemente, para que lado errar.

> **⚠️ ATENÇÃO — O threshold é uma decisão clínica, não técnica**
>
> Não existe *threshold* universalmente "certo". Ele depende do custo relativo dos erros, que é uma decisão de negócio/clínica. Nós o ancoramos em `CUSTO_FN`/`CUSTO_FP`; um hospital com outra realidade escolheria outro ponto. O papel do cientista de dados é **expor o trade-off com clareza** e deixar a decisão ser tomada com os olhos abertos — jamais aceitar o corte padrão de 0,5 cegamente.

> **📌 CHECKLIST DO CAPÍTULO 16**
>
> ✅ Entende que o *threshold* controla o *trade-off* Precisão-Recall.
>
> ✅ Sabe ancorar a escolha do *threshold* em um **custo clínico** (`CUSTO_FN`/`CUSTO_FP`).
>
> ✅ Executou o Código 16.1 e reduziu os Falsos Negativos de **9 para 2**, com o custo total caindo de 920 para 440.
>
> ✅ Reconhece que a escolha do *threshold* é uma decisão clínica, não puramente técnica.
>
> **⚠️ CRÍTICO:** Nunca aceite o *threshold* padrão sem análise. Onde os custos dos erros são assimétricos — como na medicina — ajustá-lo pode salvar vidas.

Eu finalmente me sentia no controle. Não era mais um espectador do meu modelo: era o piloto, ajustando os controles para navegar o *trade-off*. Reduzir os falsos negativos de nove para dois foi uma vitória concreta, que eu poderia explicar e justificar — porque estava ancorada num número que representava o custo real de cada erro.

Enquanto contemplava a matriz aprimorada, outro aspecto me ocorreu. Meu dataset era desbalanceado: mais benignos que malignos. E se o próprio *treinamento* estivesse enviesado para a maioria, prestando menos atenção à minoria que mais importa? Havia técnicas para reequilibrar a balança. Era hora de dar voz à minoria.
